In [1]:
!pip install spacy transformers gliner bertopic scikit-learn gensim umap-learn hdbscan pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 99.4 MB/s eta 0:00:00


In [10]:
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 79.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Сравнение NER + Topic Modeling

In [2]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
import spacy
from transformers import pipeline
from gliner import GLiNER
from bertopic import BERTopic
import gensim
from gensim import corpora
from gensim.models import LdaModel


In [3]:
texts_for_ner = [
    "Владимир Путин встретился с президентом Франции Эмманюэлем Макроном в Кремле 10 февраля 2024 года.",
    "Apple представила iPhone 15 по цене от 799 долларов на мероприятии в Купертино.",
    "Сборная России по футболу проиграла Бразилии со счетом 1:3 на стадионе Маракана.",
    "Мой email: ivan@example.com, а телефон +7 (495) 123-45-67 для связи.",
    "Иван Петров купил квартиру в Москве за 15 миллионов рублей.",
]

texts_for_topics = [
    # Технологии
    "Новый iPhone упал в воду и перестал включаться. Нужен ремонт срочно.",
    "Сервисный центр в Москве ремонтирует телефоны Apple и Samsung быстро.",
    "Замена экрана на Xiaomi Mi 11 стоит всего 3000 рублей.",
    "Ноутбук ASUS не видит Wi-Fi, что делать?",
    "Купил новый MacBook Pro, очень доволен производительностью.",
    # Кулинария
    "Как приготовить борщ со свеклой пошаговый рецепт.",
    "Пангасиус в кляре с картошкой на ужин за 30 минут.",
    "Рецепт оливье с колбасой и горошком классический.",
    "Наполеон с заварным кремом — самый вкусный торт.",
    "Плов из свинины в казане рассыпчатый, как в узбекской кухне.",
    # Спорт
    "Зенит выиграл чемпионат России по футболу.",
    "Хабиб Нурмагомедов провел бой с Макгрегором.",
    "Овечкин забил 800-ю шайбу в НХЛ.",
    "Россия проиграла Канаде в хоккейном финале.",
    "Федерер завершил карьеру после травмы колена.",
]

NER

spacy

In [11]:
nlp_spacy = spacy.load("ru_core_news_sm")

bert

In [20]:
bert_ner = pipeline("ner", model="bert-base-multilingual-cased", aggregation_strategy="simple")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly ini

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

GliNER

In [13]:
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Словарный метод

In [14]:
cities = ["Москве", "Кремле", "Купертино", "Маракана"]
names = ["Владимир", "Путин", "Эмманюэль", "Макрон", "Иван", "Петров"]
companies = ["Apple", "Xiaomi", "ASUS", "Samsung"]

In [15]:
def rule_based(text):
  entities = []
  for city in cities:
    if city in text:
      entities.append({"text":city,"label":"LOC"})
  for name in names:
    if name in text:
      entities.append({"text": name, "label": "PER"})
  for company in companies:
    if company in text:
      entities.append({"text": company, "label": "ORG"})
  return entities

In [21]:
if __name__ == '__main__':
  for text in texts_for_ner:
    print(f'Текст - {text[:100]}')
    #SPACY
    start = time.time()
    doc = nlp_spacy(text)
    spacy_time = time.time() - start
    spacy_res = [(ent.text, ent.label_) for ent in doc.ents]
    print(f'spacy - {doc}')
    print('Темы')
    print(spacy_res)
    print(f'Время - {spacy_time}')
    #BERT
    start = time.time()
    bert_res = bert_ner(text)
    bert_time = time.time() - start
    print(f'bert - {bert_res}')
    bert_res = [(ent['word'], ent['entity_group']) for ent in bert_res]
    print('Темы')
    print(bert_res)
    print(f'Время - {bert_time}')
    #GLINER
    labels = ["person","organization","location","date",'money']
    start = time.time()
    gliner_res = gliner_model.predict_entities(text,labels,threshold=0.3)
    gliner_time = time.time() - start
    print(f'gliner - {gliner_res}')
    gliner_res = [(ent['text'], ent['label']) for ent in gliner_res]
    print('Темы')
    print(gliner_res)
    print(f'Время - {gliner_time}')
    #Rule-based
    start = time.time()
    rule_res = rule_based(text)
    rule_time = time.time() - start
    rule_res = [(ent['text'], ent['label']) for ent in rule_res]
    print('Темы')
    print(rule_res)
    print(f'Время - {rule_time}')

  print(spacy_time)
  print(bert_time)
  print(gliner_time)
  print(rule_time)


Текст - Владимир Путин встретился с президентом Франции Эмманюэлем Макроном в Кремле 10 февраля 2024 года.
spacy - Владимир Путин встретился с президентом Франции Эмманюэлем Макроном в Кремле 10 февраля 2024 года.
Темы
[('Владимир Путин', 'PER'), ('Франции', 'LOC'), ('Эмманюэлем Макроном', 'PER'), ('Кремле', 'LOC')]
Время - 0.012613534927368164
bert - [{'entity_group': 'LABEL_1', 'score': np.float32(0.56785643), 'word': 'Владимир Путин в', 'start': 0, 'end': 16}, {'entity_group': 'LABEL_0', 'score': np.float32(0.5399559), 'word': '##стретился', 'start': 16, 'end': 25}, {'entity_group': 'LABEL_1', 'score': np.float32(0.5008685), 'word': 'с', 'start': 26, 'end': 27}, {'entity_group': 'LABEL_0', 'score': np.float32(0.5263965), 'word': 'президентом Франции Э', 'start': 28, 'end': 49}, {'entity_group': 'LABEL_1', 'score': np.float32(0.55526346), 'word': '##мманюэлем М', 'start': 49, 'end': 60}, {'entity_group': 'LABEL_0', 'score': np.float32(0.5067424), 'word': '##ак', 'start': 60, 'end': 6

TOPIC MODELLING

In [27]:
russian_stop_words = ['и', 'в', 'на', 'с', 'по', 'за', 'из', 'у', 'о', 'об',
                      'к', 'от', 'до', 'без', 'через', 'между', 'это', 'быть',
                      'весь', 'который', 'тот', 'свой', 'как', 'так', 'вот',
                      'лишь', 'ещё', 'уже', 'даже']

In [32]:
def run_nmf(docs):
  vectorizer = CountVectorizer(stop_words=russian_stop_words,max_features=100)
  X = vectorizer.fit_transform(docs)
  nmf = NMF(n_components=3,random_state=42)
  nmf.fit(X)
  feature_names = vectorizer.get_feature_names_out()
  topics = []
  for topic_idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-5:][::-1]]
    topics.append(f"Тема {topic_idx+1}: {', '.join(top_words)}")
  return topics


In [33]:
def run_lda(docs):
  tokenized = [doc.lower().split() for doc in docs]
  dictionary = corpora.Dictionary(tokenized)
  corpus = [dictionary.doc2bow(text) for text in tokenized]
  lda = LdaModel(corpus=corpus, id2word=dictionary,num_topics=3)
  topics = []
  for idx, topic in lda.print_topics(num_words=5):
    words = [w.split('*')[1].strip('"') for w in topic.split(' + ')]
    topics.append(f"Тема {idx+1}: {', '.join(words)}")
  return topics

In [37]:
def run_bertopic(docs):
  vectorizer_model = CountVectorizer(
    stop_words=russian_stop_words,
    min_df=1,
    max_df=0.8,
    ngram_range=(1, 2)
  )
  topic_model = BERTopic(
    language="multilingual",
    vectorizer_model=vectorizer_model,
    min_topic_size=2,
    nr_topics=3,
    calculate_probabilities=True,
    verbose=True
  )
  topics,probs = topic_model.fit_transform(docs)
  topic_info = topic_model.get_topic_info()
  print(topic_info)
  for i in range(3):
    topic_words = topic_model.get_topic(i)
    if topic_words:
        print(f"Тема {i+1}: {[w for w, _ in topic_words[:5]]}")

In [38]:
if __name__ =="__main__":
  #NMF
  start = time.time()
  nmf_topics = run_nmf(texts_for_topics)
  nmf_time = time.time() - start
  print(f'NMF - {nmf_topics}')
  print(f'Time - {nmf_time}')
  #LDA
  start = time.time()
  lda_topics = run_lda(texts_for_topics)
  lda_time = time.time() - start
  print(f'LDA - {lda_topics}')
  print(f'Time - {lda_time}')
  #BERTopic
  start = time.time()
  run_bertopic(texts_for_topics)
  bertopic_time = time.time() - start
  print(f'Time - {bertopic_time}')

2026-05-28 14:27:02,837 - BERTopic - Embedding - Transforming documents to embeddings.


NMF - ['Тема 1: новый, упал, ремонт, нужен, перестал', 'Тема 2: экрана, стоит, рублей, 3000, mi', 'Тема 3: что, wi, asus, видит, не']
Time - 0.00743556022644043
LDA - ['Тема 1: в, с, как, и, узбекской', 'Тема 2: новый, в, и, срочно., включаться.', 'Тема 3: в, с, на, футболу., экрана']
Time - 0.017137765884399414


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-28 14:27:06,834 - BERTopic - Embedding - Completed ✓
2026-05-28 14:27:06,834 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-28 14:27:06,875 - BERTopic - Dimensionality - Completed ✓
2026-05-28 14:27:06,876 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-28 14:27:06,881 - BERTopic - Cluster - Completed ✓
2026-05-28 14:27:06,882 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-28 14:27:06,889 - BERTopic - Representation - Completed ✓
2026-05-28 14:27:06,890 - BERTopic - Topic reduction - Reducing number of topics
2026-05-28 14:27:06,891 - BERTopic - Topic reduction - Number of topics (3) is equal or higher than the clustered topics(3).
2026-05-28 14:27:06,891 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-28 14:27:06,970 - BERTopic - Representation - Completed ✓


   Topic  Count                                 Name  \
0      0      5             0_новый_11_3000_11 стоит   
1      1      5            1_рецепт_30_борщ_30 минут   
2      2      5  2_800_800 шайбу_бой_бой макгрегором   

                                      Representation  \
0  [новый, 11, 3000, 11 стоит, apple, apple samsu...   
1  [рецепт, 30, борщ, 30 минут, вкусный, вкусный ...   
2  [800, 800 шайбу, бой, бой макгрегором, выиграл...   

                                 Representative_Docs  
0  [Сервисный центр в Москве ремонтирует телефоны...  
1  [Плов из свинины в казане рассыпчатый, как в у...  
2  [Зенит выиграл чемпионат России по футболу., О...  
Тема 1: ['новый', '11', '3000', '11 стоит', 'apple']
Тема 2: ['рецепт', '30', 'борщ', '30 минут', 'вкусный']
Тема 3: ['800', '800 шайбу', 'бой', 'бой макгрегором', 'выиграл']
Time - 4.152730703353882
